# WT103 Learning Curves And Scaling

This notebook separates early optimization diagnostics, matched-geometry depth scaling, and completed-run parameter scaling.

- Early diagnostics use `history.jsonl` and compare epochs 1-4, including stopped and collapsed runs.
- Matched geometry fixes `d_model=790` and compares Neo with LSTM at depths 8, 12, and 16 at epoch 4 of the same 12-epoch schedule.
- Completed scaling uses only valid completed runs with `metrics.json` and 12 observed epochs.
- Neo scaling uses the RMSNorm/all `tanh` series; LSTM uses the RMSNorm/all scaling series.
- Stopped LSTM runs diagnose smooth degradation after 40M core; they are not used for final test-PPL scaling fits.
- Both a plain power fit and a floor-aware fit `L_inf + A N^{-alpha}` are reported for completed points only.


In [ ]:
from __future__ import annotations

import json
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_ROOT = ROOT / 'runs' / 'wikitext-103-raw-v1'
TARGET_NEO_ACTIVATION = 'tanh'
TARGET_NORM = 'rmsnorm'
TARGET_NORM_PLACE = 'all'

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


def render_fig(fig):
    display(fig)
    plt.close(fig)


def note(text: str):
    display(Markdown(text))

In [ ]:
def _read_json(path: Path):
    return json.loads(path.read_text())


def _read_history(path: Path) -> pd.DataFrame:
    if not path.exists() or not path.read_text().strip():
        return pd.DataFrame(columns=['epoch', 'train_ce', 'val_ppl', 'best_val_ppl', 'global_step'])
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return pd.DataFrame(rows)


@lru_cache(maxsize=None)
def _load_cfg(cfg_path: str) -> dict:
    import yaml
    return yaml.safe_load(Path(cfg_path).read_text()) or {}


@lru_cache(maxsize=None)
def _param_breakdown_from_cfg(cfg_path: str) -> dict:
    from scripts._common import ensure_repo_root_on_path, build_model, count_params
    ensure_repo_root_on_path()
    cfg = _load_cfg(cfg_path)
    model_name = str(cfg['model_name']).strip().lower()
    model = build_model(cfg, model_name, backend_name='torch')
    total = count_params(model, backend_name='torch')
    if model_name == 'neo':
        emb = model.emb.weight.numel()
        core = sum(p.numel() for p in model.recurrent.parameters() if p.requires_grad)
        proj = sum(p.numel() for p in model.in_proj.parameters()) if hasattr(model.in_proj, 'parameters') else 0
        proj += sum(p.numel() for p in model.out_proj.parameters()) if hasattr(model.out_proj, 'parameters') else 0
        head = model.output_bias.numel() if model.output_bias is not None else 0
        if model.head is not None:
            head += sum(p.numel() for p in model.head.parameters() if p.requires_grad)
        return {'total': total, 'embeddings': emb, 'core': core, 'head_proj': proj + head}
    if model_name == 'lstm':
        emb = model.emb.weight.numel()
        core = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
        proj = sum(p.numel() for p in model.in_proj.parameters()) if hasattr(model.in_proj, 'parameters') else 0
        proj += sum(p.numel() for p in model.out_proj.parameters()) if hasattr(model.out_proj, 'parameters') else 0
        head = model.output_bias.numel() if model.output_bias is not None else 0
        if model.head is not None:
            head += sum(p.numel() for p in model.head.parameters() if p.requires_grad)
        return {'total': total, 'embeddings': emb, 'core': core, 'head_proj': proj + head}
    raise ValueError(f'Unsupported model_name for this notebook: {model_name}')


def _core_label(core_params: float) -> str:
    core_m = core_params / 1e6
    rounded = max(10, int(round(core_m / 10.0) * 10))
    return f'~{rounded}M'


def _norm_value(cfg: dict) -> str:
    return str(cfg.get('recurrent_norm', 'none')).strip().lower()


def _norm_place_value(cfg: dict) -> str:
    return str(cfg.get('recurrent_norm_place', TARGET_NORM_PLACE)).strip().lower()


def _is_target_scale_run(model_name: str, run_tag: str, cfg: dict) -> bool:
    if _norm_value(cfg) != TARGET_NORM or _norm_place_value(cfg) != TARGET_NORM_PLACE:
        return False
    if int(cfg.get('epochs', 0)) < 12 or 'ablate' in run_tag:
        return False
    if model_name == 'neo':
        activation = str(cfg.get('activation_id', '')).strip().lower()
        return activation == TARGET_NEO_ACTIVATION
    if model_name == 'lstm':
        return 'scale_rmsnorm' in run_tag
    return False


def collect_latest_scale_runs() -> pd.DataFrame:
    rows = []
    for metrics_path in RUNS_ROOT.glob('*/*/*/metrics.json'):
        run_dir = metrics_path.parent
        run_tag = run_dir.parent.name
        model_name = run_dir.parent.parent.name
        if model_name not in {'neo', 'lstm'}:
            continue
        cfg_path = run_dir / 'config.yaml'
        hist_path = run_dir / 'history.jsonl'
        if not cfg_path.exists():
            continue
        cfg = _load_cfg(str(cfg_path))
        if not _is_target_scale_run(model_name, run_tag, cfg):
            continue
        metrics = _read_json(metrics_path)
        hist = _read_history(hist_path)
        breakdown = _param_breakdown_from_cfg(str(cfg_path))
        val_series = hist['val_ppl'].astype(float) if ('val_ppl' in hist.columns and not hist.empty) else pd.Series(dtype=float)
        best_epoch = int(hist.loc[val_series.idxmin(), 'epoch']) if not val_series.empty else None
        final_epoch = int(hist['epoch'].max()) if ('epoch' in hist.columns and not hist.empty) else None
        rows.append({
            'run_dir': str(run_dir),
            'timestamp': run_dir.name,
            'run_tag': run_tag,
            'model_name': model_name,
            'config_path': str(cfg_path),
            'history_path': str(hist_path),
            'n_layers': int(cfg.get('n_layers', 0)),
            'd_model': int(cfg.get('d_model', 0)),
            'recurrent_norm': cfg.get('recurrent_norm', ''),
            'recurrent_norm_place': cfg.get('recurrent_norm_place', ''),
            'activation_id': cfg.get('activation_id', ''),
            'epochs_seen': int(len(hist)),
            'best_epoch': best_epoch,
            'final_epoch': final_epoch,
            'best_val_ppl': float(val_series.min()) if not val_series.empty else metrics.get('val_ppl'),
            'final_val_ppl': float(val_series.iloc[-1]) if not val_series.empty else metrics.get('val_ppl'),
            'final_test_ppl': metrics.get('test_ppl'),
            'best_test_ppl': metrics.get('best_ckpt_test_ppl_torch', metrics.get('test_ppl')),
            'total_params': breakdown['total'],
            'embedding_params': breakdown['embeddings'],
            'core_params': breakdown['core'],
            'head_proj_params': breakdown['head_proj'],
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.sort_values(['run_tag', 'timestamp']).groupby('run_tag', as_index=False).tail(1).copy()
    df['core_label'] = df['core_params'].map(_core_label)
    df['core_params_m'] = df['core_params'] / 1e6
    df['total_params_m'] = df['total_params'] / 1e6
    df['embedding_params_m'] = df['embedding_params'] / 1e6
    return df.sort_values(['model_name', 'core_params']).reset_index(drop=True)


scale_df = collect_latest_scale_runs()
if scale_df.empty:
    note('**No completed scaling runs found yet.** Run the scaling experiments first, then re-run this notebook.')
scale_df

EARLY_EPOCHS = (1, 2, 3, 4)
DIAGNOSTIC_RUN_TAGS = {
    # LSTM completed scaling points plus stopped large-depth diagnostics.
    'wt103_lstm_20m_scale_rmsnorm',
    'wt103_lstm_30m_scale_rmsnorm',
    'wt103_lstm_50m_scale_rmsnorm',
    'wt103_lstm_60m_rmsnorm',
    'wt103_lstm_70m_rmsnorm',
    'wt103_lstm_80m_rmsnorm',
    'wt103_lstm_90m_rmsnorm',
    # Neo completed tanh scaling points plus the 80M-core collapse diagnostic.
    'wt103_neo_20m_scale_rmsnorm_tanh',
    'wt103_neo_30m_scale_rmsnorm_tanh',
    'wt103_neo_50m_scale_rmsnorm_tanh',
    'wt103_neo_70m_tanh_rmsnorm',
    'wt103_neo_80m_tanh_rmsnorm',
    'wt103_neo_90m_tanh_rmsnorm',
}


def _history_status(run_tag: str, hist: pd.DataFrame, metrics_path: Path, cfg: dict) -> str:
    epochs_seen = int(len(hist))
    target_epochs = int(cfg.get('epochs', 0) or 0)
    val_series = hist['val_ppl'].astype(float) if ('val_ppl' in hist.columns and not hist.empty) else pd.Series(dtype=float)
    if not val_series.empty and float(val_series.max()) > 1000:
        return 'collapsed'
    if metrics_path.exists() and target_epochs and epochs_seen >= target_epochs:
        return 'completed'
    if run_tag.startswith('wt103_lstm_') and epochs_seen <= 4:
        return 'stopped_degraded'
    return 'partial'


def collect_latest_diagnostic_runs() -> pd.DataFrame:
    rows = []
    for hist_path in RUNS_ROOT.glob('*/*/*/history.jsonl'):
        run_dir = hist_path.parent
        run_tag = run_dir.parent.name
        if run_tag not in DIAGNOSTIC_RUN_TAGS:
            continue
        model_name = run_dir.parent.parent.name
        if model_name not in {'neo', 'lstm'}:
            continue
        cfg_path = run_dir / 'config.yaml'
        if not cfg_path.exists():
            continue
        cfg = _load_cfg(str(cfg_path))
        if _norm_value(cfg) != TARGET_NORM or _norm_place_value(cfg) != TARGET_NORM_PLACE:
            continue
        if model_name == 'neo' and str(cfg.get('activation_id', '')).strip().lower() != TARGET_NEO_ACTIVATION:
            continue
        hist = _read_history(hist_path)
        if hist.empty:
            continue
        breakdown = _param_breakdown_from_cfg(str(cfg_path))
        metrics_path = run_dir / 'metrics.json'
        val_series = hist['val_ppl'].astype(float) if 'val_ppl' in hist.columns else pd.Series(dtype=float)
        row = {
            'run_dir': str(run_dir),
            'timestamp': run_dir.name,
            'run_tag': run_tag,
            'model_name': model_name,
            'config_path': str(cfg_path),
            'history_path': str(hist_path),
            'n_layers': int(cfg.get('n_layers', 0)),
            'd_model': int(cfg.get('d_model', 0)),
            'epochs_seen': int(len(hist)),
            'status': _history_status(run_tag, hist, metrics_path, cfg),
            'best_observed_val_ppl': float(val_series.min()) if not val_series.empty else np.nan,
            'last_observed_val_ppl': float(val_series.iloc[-1]) if not val_series.empty else np.nan,
            'total_params': breakdown['total'],
            'core_params': breakdown['core'],
        }
        for epoch in EARLY_EPOCHS:
            matched = hist.loc[hist['epoch'].astype(int) == epoch, 'val_ppl'] if 'epoch' in hist.columns else pd.Series(dtype=float)
            row[f'epoch{epoch}_val_ppl'] = float(matched.iloc[0]) if not matched.empty else np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.sort_values(['run_tag', 'timestamp']).groupby('run_tag', as_index=False).tail(1).copy()
    df['core_label'] = df['core_params'].map(_core_label)
    df['core_params_m'] = df['core_params'] / 1e6
    df['total_params_m'] = df['total_params'] / 1e6
    return df.sort_values(['model_name', 'core_params_m']).reset_index(drop=True)


diagnostic_df = collect_latest_diagnostic_runs()
diagnostic_df


## Overview

The notebook is organized into three different claims.

1. **Early optimization diagnostics:** compare validation PPL at epochs 1-4 across core sizes. This includes stopped runs and shows failure modes: Neo's sharp 80M-core collapse and LSTM's smooth degradation after 40M core.
2. **Matched-geometry depth scaling:** hold `d_model=790` fixed and compare Neo with LSTM at depths 8, 12, and 16. This tests whether narrow width plus increasing depth is sufficient to reproduce LSTM's degradation.
3. **Completed scaling fits:** use only completed, valid 12-epoch runs with final metrics. These plots are fits over observed completed points, not evidence from the stopped diagnostics.

The recurrent-core parameter axis is used for capacity scaling. The matched-geometry section instead uses layer count on the x-axis because width and depth are deliberately held equal across model families.


## Early-Epoch Diagnostics

This section uses `history.jsonl` directly, so stopped and collapsed runs are visible. These points diagnose optimization behavior, but they are not used in the completed-run test-PPL scaling fit.


In [ ]:
epoch_cols = [f'epoch{e}_val_ppl' for e in EARLY_EPOCHS]
summary_cols = [
    'model_name', 'run_tag', 'status', 'n_layers', 'd_model', 'epochs_seen', 'core_params_m',
    *epoch_cols, 'best_observed_val_ppl', 'last_observed_val_ppl'
]

if diagnostic_df.empty:
    note('**No diagnostic histories found.**')
else:
    display(diagnostic_df[summary_cols].sort_values(['model_name', 'core_params_m']).reset_index(drop=True))

    epoch_colors = {1: '#6E7F91', 2: '#4F79A7', 3: '#8A6FB0', 4: '#A54E58'}
    status_markers = {'completed': 'o', 'stopped_degraded': 's', 'collapsed': 'X', 'partial': '^'}

    for model_name in ['neo', 'lstm']:
        sub = diagnostic_df[diagnostic_df['model_name'] == model_name].sort_values('core_params_m')
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for epoch in EARLY_EPOCHS:
            col = f'epoch{epoch}_val_ppl'
            points = sub[np.isfinite(sub[col])]
            if points.empty:
                continue
            ax.plot(
                points['core_params_m'],
                points[col],
                color=epoch_colors[epoch],
                lw=1.5,
                alpha=0.75,
                label=f'Epoch {epoch}',
            )
            for _, row in points.iterrows():
                ax.scatter(
                    row['core_params_m'],
                    row[col],
                    color=epoch_colors[epoch],
                    marker=status_markers.get(row['status'], 'o'),
                    s=62,
                    edgecolor='black',
                    linewidth=0.45,
                    zorder=4,
                )
        ax.set_title(f'{model_name.upper()} Early-Epoch Optimization Diagnostics')
        ax.set_xlabel('Recurrent Core Params (Millions)')
        ax.set_ylabel('Validation PPL At Matched Epochs (log scale)')
        ax.set_yscale('log')
        ax.grid(True, which='both', alpha=0.25)
        ax.set_xlim(max(1, sub['core_params_m'].min() * 0.85), sub['core_params_m'].max() * 1.12)
        ax.legend(frameon=True, fontsize=9)
        plt.tight_layout()
        render_fig(fig)

    status_handles = [
        Line2D([0], [0], marker=marker, color='w', markerfacecolor='#777777', markeredgecolor='black', markersize=8, lw=0, label=status.replace('_', ' '))
        for status, marker in status_markers.items()
        if status in set(diagnostic_df['status'])
    ]
    if status_handles:
        fig, ax = plt.subplots(figsize=(7, 1.2))
        ax.axis('off')
        ax.legend(handles=status_handles, loc='center', ncol=len(status_handles), frameon=True)
        render_fig(fig)


## Matched-Geometry Depth Scaling

This section isolates the width-depth hypothesis from parameter-budget scaling.

- Both model families use `d_model=790`, RMSNorm/all, the streaming regime, and the same 12-epoch cosine schedule.
- Neo uses the tanh baseline.
- Depths 8, 12, and 16 match the LSTM 40M-, 60M-, and 80M-core geometries.
- Validation PPL is read at epoch 4 for every run, including runs that continued further.

The comparison asks whether fixed narrow width plus increasing depth is sufficient to cause degradation. It does not compare equal parameter counts: Neo has fewer recurrent parameters at the same geometry.


In [ ]:
MATCHED_GEOMETRY_WIDTH = 790
MATCHED_GEOMETRY_DEPTHS = (8, 12, 16)
MATCHED_GEOMETRY_EPOCH = 4
MATCHED_GEOMETRY_TAGS = {
    ('neo', 8): 'wt103_neo_d790_l8_tanh_rmsnorm_geom_lstm40m',
    ('neo', 12): 'wt103_neo_d790_l12_tanh_rmsnorm_geom_lstm60m',
    ('neo', 16): 'wt103_neo_d790_l16_tanh_rmsnorm_geom_lstm80m',
    ('lstm', 8): 'wt103_lstm_50m_scale_rmsnorm',
    ('lstm', 12): 'wt103_lstm_70m_rmsnorm',
    ('lstm', 16): 'wt103_lstm_90m_rmsnorm',
}


def collect_matched_geometry_runs() -> pd.DataFrame:
    rows = []
    for (model_name, depth), run_tag in MATCHED_GEOMETRY_TAGS.items():
        candidates = []
        for history_path in sorted((RUNS_ROOT / model_name / run_tag).glob('*/history.jsonl')):
            run_dir = history_path.parent
            cfg_path = run_dir / 'config.yaml'
            if not cfg_path.exists():
                continue
            cfg = _load_cfg(str(cfg_path))
            if int(cfg.get('epochs', 0)) != 12:
                continue
            if int(cfg.get('d_model', 0)) != MATCHED_GEOMETRY_WIDTH:
                continue
            if int(cfg.get('n_layers', 0)) != depth:
                continue
            if _norm_value(cfg) != TARGET_NORM or _norm_place_value(cfg) != TARGET_NORM_PLACE:
                continue
            if model_name == 'neo' and str(cfg.get('activation_id', '')).strip().lower() != TARGET_NEO_ACTIVATION:
                continue

            history = _read_history(history_path)
            epoch_row = history.loc[history['epoch'].astype(int) == MATCHED_GEOMETRY_EPOCH]
            if epoch_row.empty:
                continue
            epoch_row = epoch_row.iloc[0]
            breakdown = _param_breakdown_from_cfg(str(cfg_path))
            candidates.append({
                'model_name': model_name,
                'depth': depth,
                'd_model': int(cfg['d_model']),
                'run_tag': run_tag,
                'timestamp': run_dir.name,
                'run_dir': str(run_dir),
                'configured_epochs': int(cfg['epochs']),
                'epochs_seen': int(len(history)),
                'epoch4_train_ce': float(epoch_row['train_ce']),
                'epoch4_val_ppl': float(epoch_row['val_ppl']),
                'core_params_m': breakdown['core'] / 1e6,
                'total_params_m': breakdown['total'] / 1e6,
            })
        if not candidates:
            raise RuntimeError(f'No valid matched-geometry run found for {model_name} depth {depth}.')
        rows.append(sorted(candidates, key=lambda row: row['timestamp'])[-1])

    geometry = pd.DataFrame(rows).sort_values(['model_name', 'depth']).reset_index(drop=True)
    expected_pairs = set(MATCHED_GEOMETRY_TAGS)
    observed_pairs = set(zip(geometry['model_name'], geometry['depth']))
    assert observed_pairs == expected_pairs
    assert len(geometry) == 6
    assert set(geometry['d_model']) == {MATCHED_GEOMETRY_WIDTH}
    assert set(geometry['configured_epochs']) == {12}
    assert np.isfinite(geometry['epoch4_val_ppl']).all()
    geometry['delta_vs_depth8'] = geometry.groupby('model_name')['epoch4_val_ppl'].transform(lambda values: values - values.iloc[0])
    return geometry


geometry_df = collect_matched_geometry_runs()
geometry_columns = [
    'model_name', 'depth', 'd_model', 'configured_epochs', 'epochs_seen',
    'core_params_m', 'total_params_m', 'epoch4_train_ce', 'epoch4_val_ppl',
    'delta_vs_depth8', 'run_tag', 'timestamp',
]
display(geometry_df[geometry_columns])


In [ ]:
geometry_styles = {
    'neo': {'color': '#4F79A7', 'marker': 'o', 'linestyle': '-', 'label': 'Neo'},
    'lstm': {'color': '#A54E58', 'marker': 's', 'linestyle': '--', 'label': 'LSTM'},
}

fig, ax = plt.subplots(figsize=(9.2, 5.7))
for model_name in ('neo', 'lstm'):
    sub = geometry_df[geometry_df['model_name'] == model_name].sort_values('depth')
    style = geometry_styles[model_name]
    ax.plot(
        sub['depth'],
        sub['epoch4_val_ppl'],
        color=style['color'],
        marker=style['marker'],
        linestyle=style['linestyle'],
        linewidth=2.2,
        markersize=7.5,
        markeredgecolor='#222222',
        markeredgewidth=0.6,
        label=style['label'],
    )
    for _, row in sub.iterrows():
        ax.annotate(
            f"{row['epoch4_val_ppl']:.2f}",
            (row['depth'], row['epoch4_val_ppl']),
            textcoords='offset points',
            xytext=(0, 9),
            ha='center',
            color=style['color'],
            fontsize=9,
            fontweight='semibold',
        )

ax.set_title('WT103 Matched-Geometry Depth Comparison', loc='left', fontsize=14, pad=28)
ax.text(
    0.0,
    1.015,
    'd_model=790 | RMSNorm/all | epoch 4 of a 12-epoch schedule | seed 42',
    transform=ax.transAxes,
    ha='left',
    va='bottom',
    fontsize=9.5,
    color='#555555',
)
ax.set_xlabel('Recurrent Layers')
ax.set_ylabel('Validation PPL At Epoch 4')
ax.set_xticks(MATCHED_GEOMETRY_DEPTHS)
ax.set_xlim(7.2, 16.8)
ax.grid(axis='y', alpha=0.25)
ax.grid(axis='x', visible=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=True, loc='upper left')
plt.tight_layout()
render_fig(fig)

neo = geometry_df[geometry_df['model_name'] == 'neo'].set_index('depth')
lstm = geometry_df[geometry_df['model_name'] == 'lstm'].set_index('depth')
neo_delta = float(neo.loc[16, 'epoch4_val_ppl'] - neo.loc[8, 'epoch4_val_ppl'])
lstm_delta = float(lstm.loc[16, 'epoch4_val_ppl'] - lstm.loc[8, 'epoch4_val_ppl'])
neo_last_step = float(neo.loc[16, 'epoch4_val_ppl'] - neo.loc[12, 'epoch4_val_ppl'])
note(
    f"**Observed at epoch 4:** Neo changes from {neo.loc[8, 'epoch4_val_ppl']:.2f} to "
    f"{neo.loc[16, 'epoch4_val_ppl']:.2f} ({neo_delta:+.2f}), while LSTM changes from "
    f"{lstm.loc[8, 'epoch4_val_ppl']:.2f} to {lstm.loc[16, 'epoch4_val_ppl']:.2f} "
    f"({lstm_delta:+.2f}). Neo's depth-12 to depth-16 change is {neo_last_step:+.2f}, "
    "so the evidence is the absence of LSTM-like degradation rather than a claim of continued large gains. "
    "With one seed, this shows that narrow width plus depth is not sufficient to explain the LSTM trend."
)


## Completed-Run Parameter Scaling

The remaining sections return to recurrent-core parameter scaling. They use only completed runs and do not include the stopped matched-geometry probes above.


In [ ]:
COLOR_MAP = {'~10M': '#5C86B0', '~20M': '#A54E58', '~40M': '#6F9F8E', '~50M': '#8A6FB0', '~60M': '#C0843E', '~70M': '#5F8F8B', '~80M': '#7A7A7A'}

if not scale_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, model_name in zip(axes, ['neo', 'lstm']):
        sub = scale_df[scale_df['model_name'] == model_name].sort_values('core_params')
        for _, row in sub.iterrows():
            hist = _read_history(Path(row['history_path']))
            if hist.empty or 'val_ppl' not in hist.columns:
                continue
            color = COLOR_MAP.get(row['core_label'], '#333333')
            label = f"{row['core_label']} core ({row['core_params_m']:.1f}M)"
            ax.plot(hist['epoch'], hist['val_ppl'], marker='o', lw=2, color=color, label=label)
            best_idx = hist['val_ppl'].astype(float).idxmin()
            ax.scatter(
                hist.loc[best_idx, 'epoch'],
                hist.loc[best_idx, 'val_ppl'],
                color=color,
                s=90,
                marker='*',
                edgecolor='black',
                linewidth=0.6,
            )
        ax.set_title(model_name.upper())
        ax.set_xlabel('Epoch')
        ax.set_yscale('log')
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(frameon=True)
        else:
            ax.text(0.5, 0.5, 'No finished curves yet', ha='center', va='center', transform=ax.transAxes)
    axes[0].set_ylabel('Validation PPL (log scale)')
    fig.suptitle('WT103 Completed-Run Learning Curves', y=1.03, fontsize=14)
    plt.tight_layout()
    render_fig(fig)


In [ ]:
summary_cols = [
    'model_name', 'run_tag', 'activation_id', 'n_layers', 'd_model', 'epochs_seen', 'core_params_m',
    'best_epoch', 'best_val_ppl', 'best_test_ppl', 'final_test_ppl', 'total_params_m'
]
if not scale_df.empty:
    display(scale_df[summary_cols].sort_values(['model_name', 'core_params_m']).reset_index(drop=True))

In [ ]:
def _clean_xy(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    return x[mask], y[mask]


def fit_loglog_power(x, y):
    x, y = _clean_xy(x, y)
    if x.size < 2:
        return None
    slope, intercept = np.polyfit(np.log(x), np.log(y), 1)
    pred = np.exp(intercept) * np.power(x, slope)
    rmse = float(np.sqrt(np.mean((pred - y) ** 2)))
    return {
        'kind': 'plain',
        'slope': float(slope),
        'intercept': float(intercept),
        'alpha_if_decreasing': float(-slope),
        'rmse': rmse,
        'predict': lambda x_new: np.exp(intercept) * np.power(np.asarray(x_new, dtype=float), slope),
    }


def fit_power_with_floor(x, y, grid_size=400):
    x, y = _clean_xy(x, y)
    if x.size < 3:
        return None
    floor_max = float(np.min(y) * 0.999)

    def _candidate(floor):
        shifted = y - floor
        if np.any(shifted <= 0):
            return None
        slope, intercept = np.polyfit(np.log(x), np.log(shifted), 1)
        pred = floor + np.exp(intercept) * np.power(x, slope)
        rmse = float(np.sqrt(np.mean((pred - y) ** 2)))
        return {
            'kind': 'floor',
            'floor': float(floor),
            'slope': float(slope),
            'intercept': float(intercept),
            'alpha_if_decreasing': float(-slope),
            'rmse': rmse,
            'predict': lambda x_new, floor=floor, intercept=intercept, slope=slope: floor + np.exp(intercept) * np.power(np.asarray(x_new, dtype=float), slope),
        }

    if x.size == 3:
        def _slope_gap(floor):
            shifted = y - floor
            if np.any(shifted <= 0):
                return np.nan
            z = np.log(shifted)
            s01 = (z[0] - z[1]) / (np.log(x[0]) - np.log(x[1]))
            s12 = (z[1] - z[2]) / (np.log(x[1]) - np.log(x[2]))
            return float(s01 - s12)

        grid = np.linspace(0.0, floor_max, 4096)
        prev_f = None
        prev_v = None
        for floor in grid:
            val = _slope_gap(float(floor))
            if not np.isfinite(val):
                continue
            if prev_v is not None and np.sign(prev_v) != np.sign(val):
                lo, hi = prev_f, float(floor)
                vlo, vhi = prev_v, val
                for _ in range(80):
                    mid = 0.5 * (lo + hi)
                    vmid = _slope_gap(mid)
                    if not np.isfinite(vmid):
                        break
                    if np.sign(vlo) == np.sign(vmid):
                        lo, vlo = mid, vmid
                    else:
                        hi, vhi = mid, vmid
                exact = _candidate(0.5 * (lo + hi))
                if exact is not None:
                    return exact
            prev_f, prev_v = float(floor), val

    best = None
    for floor in np.linspace(0.0, float(np.min(y) * 0.98), grid_size):
        candidate = _candidate(float(floor))
        if candidate is None:
            continue
        if best is None or candidate['rmse'] < best['rmse']:
            best = candidate
    return best


def find_family_crossover(fit_a, fit_b, x_lo, x_hi, grid_size=4096):
    if fit_a is None or fit_b is None:
        return None
    x_grid = np.geomspace(float(x_lo), float(x_hi), grid_size)
    diff = fit_a['predict'](x_grid) - fit_b['predict'](x_grid)
    idx = np.where(np.sign(diff[:-1]) != np.sign(diff[1:]))[0]
    if idx.size == 0:
        return None
    j = int(idx[0])
    x0, x1 = float(x_grid[j]), float(x_grid[j + 1])
    d0, d1 = float(diff[j]), float(diff[j + 1])
    if d1 == d0:
        x_cross = x0
    else:
        x_cross = x0 + (x1 - x0) * (-d0) / (d1 - d0)
    y_cross = float(fit_a['predict']([x_cross])[0])
    return {'x': x_cross, 'y': y_cross}


fit_rows = []
if not scale_df.empty:
    for model_name, sub in scale_df.groupby('model_name'):
        fit = fit_loglog_power(sub['core_params_m'].values, sub['best_test_ppl'].values)
        floor_fit = fit_power_with_floor(sub['core_params_m'].values, sub['best_test_ppl'].values)
        x_min = float(np.min(sub['core_params_m'].values))
        x_max = float(np.max(sub['core_params_m'].values))
        fit_rows.append({
            'model_name': model_name,
            'n_points': int(len(sub)),
            'plain_fit_available': fit is not None,
            'plain_alpha': np.nan if fit is None else fit['alpha_if_decreasing'],
            'plain_rmse': np.nan if fit is None else fit['rmse'],
            'floor_fit_available': floor_fit is not None,
            'floor_linf': np.nan if floor_fit is None else floor_fit['floor'],
            'floor_alpha': np.nan if floor_fit is None else floor_fit['alpha_if_decreasing'],
            'floor_rmse': np.nan if floor_fit is None else floor_fit['rmse'],
            'observed_start_m': x_min,
            'observed_end_m': x_max,
        })
fit_df = pd.DataFrame(fit_rows)
fit_df

In [ ]:
def plot_fit_family(fit_kind: str, title: str):
    family_colors = {'neo': '#4F79A7', 'lstm': '#A54E58'}
    line_width = 1.2 if fit_kind == 'plain' else 1.35
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    ax_normal, ax_log = axes

    def _fit_for(sub):
        x = sub['core_params_m'].values
        y = sub['best_test_ppl'].values
        return fit_loglog_power(x, y) if fit_kind == 'plain' else fit_power_with_floor(x, y)

    range_lines = []
    fit_by_model = {}
    crossover = None
    for model_name, sub in scale_df.groupby('model_name'):
        sub = sub.sort_values('core_params_m')
        color = family_colors[model_name]
        fit = _fit_for(sub)
        fit_by_model[model_name] = fit
        x_min, x_max = sub['core_params_m'].min(), sub['core_params_m'].max()
        x_left = np.geomspace(x_min * 0.75, x_min, 64)
        x_obs = np.geomspace(x_min, x_max, 180)
        x_right = np.geomspace(x_max, x_max * 1.6, 64)
        range_lines.append(f"{model_name.upper()}: {x_min:.2f}M to {x_max:.2f}M")

        for ax in axes:
            ax.plot(sub['core_params_m'], sub['best_test_ppl'], marker='o', lw=0, ms=7, color=color)
            if fit is not None:
                ax.plot(x_left, fit['predict'](x_left), lw=line_width, ls=(0, (3, 3)), color=color, alpha=0.45)
                ax.plot(x_obs, fit['predict'](x_obs), lw=line_width, color=color, alpha=0.92)
                ax.plot(x_right, fit['predict'](x_right), lw=line_width, ls=(0, (3, 3)), color=color, alpha=0.45)
            if fit_kind == 'floor' and fit is not None:
                ax.axhline(fit['floor'], lw=0.8, ls=':', color=color, alpha=0.7)
            for x_bound in (x_min, x_max):
                ax.axvline(x_bound, lw=0.8, ls=':', color='#777777', alpha=0.6)

    if 'neo' in fit_by_model and 'lstm' in fit_by_model:
        x_lo = float(scale_df['core_params_m'].min() * 0.75)
        x_hi = float(scale_df['core_params_m'].max() * 3.0)
        crossover = find_family_crossover(fit_by_model['neo'], fit_by_model['lstm'], x_lo, x_hi)
        if crossover is not None:
            for ax in axes:
                ax.axvline(crossover['x'], lw=1.0, ls='-.', color='#444444', alpha=0.8)
                ax.scatter([crossover['x']], [crossover['y']], s=26, color='#222222', zorder=6)
                ax.annotate(f"{crossover['x']:.2f}M", (crossover['x'], crossover['y']), textcoords='offset points', xytext=(6, 6), fontsize=8, color='#222222')

    note_text = 'Observed fit range\n' + '\n'.join(range_lines)
    if crossover is not None:
        note_text += f"\nCrossover: {crossover['x']:.2f}M"
    if fit_kind == 'floor':
        floor_lines = [
            f"{row['model_name'].upper()} floor: {row['floor_linf']:.2f}"
            for _, row in fit_df.iterrows()
            if np.isfinite(row['floor_linf'])
        ]
        if floor_lines:
            note_text += '\n' + '\n'.join(floor_lines)
    for ax in axes:
        ax.text(0.03, 0.04, note_text, transform=ax.transAxes, fontsize=8.5, va='bottom', ha='left', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#CCCCCC', alpha=0.9))

    ax_normal.set_title('Normal Scale')
    ax_normal.set_xlabel('Recurrent Core Params (Millions)')
    ax_normal.set_ylabel('Best-Checkpoint Test PPL')

    ax_log.set_title('Log Scale')
    ax_log.set_xlabel('Recurrent Core Params (Millions, log scale)')
    ax_log.set_ylabel('Best-Checkpoint Test PPL (log scale)')
    ax_log.set_xscale('log')
    ax_log.set_yscale('log')

    legend_handles = [
        Line2D([0], [0], color='#555555', lw=line_width, label='Observed-range fit'),
        Line2D([0], [0], color='#555555', lw=line_width, ls=(0, (3, 3)), label='Extrapolated fit'),
        Line2D([0], [0], color='#777777', lw=0.8, ls=':', label='Observed-range boundary'),
        Line2D([0], [0], color='#444444', lw=1.0, ls='-.', label='Crossover param'),
        Line2D([0], [0], marker='o', color='#222222', markersize=5, lw=0, label='Crossover point'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=family_colors['neo'], markersize=7, label='Neo'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=family_colors['lstm'], markersize=7, label='LSTM'),
    ]
    ax_normal.legend(handles=legend_handles, frameon=True, fontsize=9)
    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    render_fig(fig)


if not scale_df.empty:
    plot_fit_family('plain', 'WT103 Power Fit')
    plot_fit_family('floor', 'WT103 Floor-Aware Fit')
else:
    note('**Scaling plot skipped** because no finished scaling runs were found.')

## Notes

- The early-epoch diagnostic plot includes stopped and collapsed runs from `history.jsonl`.
- The matched-geometry comparison uses only `d_model=790` runs configured with the same 12-epoch schedule and reads validation PPL at epoch 4.
- The older Neo `d790/l12` run configured for four total epochs is excluded because its cosine schedule is not comparable.
- The geometry comparison is one-seed evidence about optimization behavior; it is not a parameter-efficiency comparison.
- The completed scaling fits use only valid completed points with `metrics.json`.
- Solid fit segments mark the observed completed-run range.
- Faded dashed segments are extrapolations beyond the completed-run range.
- The floor-aware fit is useful when the curve visibly bends toward an asymptote.
- The plain power fit is still useful as a simple slope summary, but it is a fit over completed observations rather than a reliable prediction once degradation/collapse appears.
